In [0]:
#Initialize Spark


from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DistributedDataProcessing") \
    .getOrCreate()

print("Spark session is ready!")
print("Spark version:", spark.version)

Spark session is ready!
Spark version: 4.2.0


In [0]:
#Create the original dataset


data = [
    (101, "Rahul", "Laptop", 55000, 1),
    (102, "Priya", "Mobile", 20000, 2),
    (103, "Karthik", "Tablet", 15000, 1),
    (104, "Sneha", "Laptop", 55000, 2),
    (105, None, "Mobile", 20000, 1),
    (106, "Anil", None, 10000, 2),
    (107, "Rahul", "Mobile", 20000, 2),
    (107, "Rahul", "Mobile", 20000, 2),
    (108, "Sneha", "Headphones", -2000, 1),
    (109, "Priya", "Tablet", 15000, 2),
    (110, "Anil", "Laptop", 55000, 1)
]

columns = [
    "order_id",
    "customer_name",
    "product",
    "price",
    "quantity"
]

df = spark.createDataFrame(data, columns)

display(df)

print("Original records:", df.count())

order_id,customer_name,product,price,quantity
101,Rahul,Laptop,55000,1
102,Priya,Mobile,20000,2
103,Karthik,Tablet,15000,1
104,Sneha,Laptop,55000,2
105,null,Mobile,20000,1
106,Anil,null,10000,2
107,Rahul,Mobile,20000,2
107,Rahul,Mobile,20000,2
108,Sneha,Headphones,-2000,1
109,Priya,Tablet,15000,2


Original records: 11


In [0]:
#Explore the dataset

from pyspark.sql.functions import col, sum as spark_sum

# Display the first five rows
df.show(5)

# Check column names and data types
df.printSchema()

# Display basic statistics
df.describe().show()

# Count missing values in each column
df.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in df.columns
]).show()


+--------+-------------+-------+-----+--------+
|order_id|customer_name|product|price|quantity|
+--------+-------------+-------+-----+--------+
|     101|        Rahul| Laptop|55000|       1|
|     102|        Priya| Mobile|20000|       2|
|     103|      Karthik| Tablet|15000|       1|
|     104|        Sneha| Laptop|55000|       2|
|     105|         NULL| Mobile|20000|       1|
+--------+-------------+-------+-----+--------+
only showing top 5 rows
root
 |-- order_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- price: long (nullable = true)
 |-- quantity: long (nullable = true)

+-------+------------------+-------------+----------+------------------+------------------+
|summary|          order_id|customer_name|   product|             price|          quantity|
+-------+------------------+-------------+----------+------------------+------------------+
|  count|                11|           10|        10|              

In [0]:
#Clean the dataset

from pyspark.sql.functions import col

# Remove rows with missing customer names or products
clean_df = df.dropna(
    subset=["customer_name", "product"]
)

# Remove invalid prices and quantities
clean_df = clean_df.filter(
    (col("price") > 0) &
    (col("quantity") > 0)
)

# Remove duplicate order IDs
clean_df = clean_df.dropDuplicates(["order_id"])

display(clean_df)

print("Original records:", df.count())
print("Cleaned records:", clean_df.count())


order_id,customer_name,product,price,quantity
101,Rahul,Laptop,55000,1
102,Priya,Mobile,20000,2
103,Karthik,Tablet,15000,1
104,Sneha,Laptop,55000,2
107,Rahul,Mobile,20000,2
109,Priya,Tablet,15000,2
110,Anil,Laptop,55000,1


Original records: 11
Cleaned records: 7


In [0]:
#Validate the cleaned data


from pyspark.sql.functions import col, sum as spark_sum

# Verify missing values
clean_df.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in clean_df.columns
]).show()

# Verify unique order IDs
print(
    "Unique order IDs:",
    clean_df.select("order_id").distinct().count()
)


+--------+-------------+-------+-----+--------+
|order_id|customer_name|product|price|quantity|
+--------+-------------+-------+-----+--------+
|       0|            0|      0|    0|       0|
+--------+-------------+-------+-----+--------+

Unique order IDs: 7


In [0]:
#Calculate total sales for each order


from pyspark.sql.functions import col

sales_df = clean_df.withColumn(
    "total_amount",
    col("price") * col("quantity")
)

display(sales_df)


order_id,customer_name,product,price,quantity,total_amount
101,Rahul,Laptop,55000,1,55000
102,Priya,Mobile,20000,2,40000
103,Karthik,Tablet,15000,1,15000
104,Sneha,Laptop,55000,2,110000
107,Rahul,Mobile,20000,2,40000
109,Priya,Tablet,15000,2,30000
110,Anil,Laptop,55000,1,55000


In [0]:
# Product-wise sales analysis


from pyspark.sql.functions import sum as spark_sum, count

product_sales = sales_df.groupBy("product").agg(
    spark_sum("total_amount").alias("total_sales"),
    count("order_id").alias("total_orders")
)

display(
    product_sales.orderBy(
        col("total_sales").desc()
    )
)


product,total_sales,total_orders
Laptop,220000,3
Mobile,80000,2
Tablet,45000,2


In [0]:
#  Customer-wise sales analysis

customer_sales = sales_df.groupBy("customer_name").agg(
    spark_sum("total_amount").alias("total_spent"),
    count("order_id").alias("total_orders")
)

display(
    customer_sales.orderBy(
        col("total_spent").desc()
    )
)

customer_name,total_spent,total_orders
Sneha,110000,1
Rahul,95000,2
Priya,70000,2
Anil,55000,1
Karthik,15000,1


In [0]:
# Simulate the same data using a Spark DataFrame
prices = [100, 200, 300, 400, 500, 600]

prices_df = spark.createDataFrame(
    [(price,) for price in prices],
    ["price"]
)

display(prices_df)

print("Number of records:", prices_df.count())

price
100
200
300
400
500
600


Number of records: 6


In [0]:
# Perform transformation and filtering


from pyspark.sql.functions import col

# Transformation: apply a 10% discount
discounted_df = prices_df.select(
    (col("price") * 0.90).alias("discounted_price")
)

display(discounted_df)

# Filter prices greater than 300
filtered_df = prices_df.filter(
    col("price") > 300
)

display(filtered_df)

print("Filtered records:", filtered_df.count())

discounted_price
90.0
180.0
270.0
360.0
450.0
540.0


price
400
500
600


Filtered records: 3


In [0]:
# Save sales data as Parquet


(
    sales_df.write
    .mode("overwrite")
    .format("parquet")
    .save("/tmp/processed_sales")
)

print("Sales data saved successfully!")

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-6683569393154554>, line 8
      1 # Save sales data as Parquet
      4 (
      5     sales_df.write
      6     .mode("overwrite")
      7     .format("parquet")
----> 8     .save("/tmp/processed_sales")
      9 )
     11 print("Sales data saved successfully!")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:794, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    792     self.format(format)
    793 self._write.path = path
--> 794 _, _, ei = self._spark.client.execute_command(
    795     self._write.command(self._spark.client), self._write.observations
    796 )
    797 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1812, in SparkConnectClient.execute_command(self, command, observations, extra_req

In [0]:
display(spark.sql("SHOW CATALOGS"))

catalog
samples
system
workspace


In [0]:
display(spark.sql("SHOW SCHEMAS IN workspace"))

databaseName
bronze
default
gold
information_schema
silver


In [0]:
display(
    spark.sql("SHOW VOLUMES IN workspace.bronze")
)

database,volume_name


In [0]:
# Check existing tables


display(
    spark.sql("SHOW TABLES IN workspace.bronze")
)

database,tableName,isTemporary
bronze,customers,false
bronze,products,false
bronze,sales,false
bronze,stores,false


In [0]:
# Save your processed sales data


sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.pyspark_processed_sales")

print("Processed sales data saved successfully!")


Processed sales data saved successfully!


In [0]:
# Verify the saved table

saved_df = spark.table(
    "workspace.silver.pyspark_processed_sales"
)

display(saved_df)

print("Saved records:", saved_df.count())


order_id,customer_name,product,price,quantity,total_amount
101,Rahul,Laptop,55000,1,55000
102,Priya,Mobile,20000,2,40000
103,Karthik,Tablet,15000,1,15000
104,Sneha,Laptop,55000,2,110000
107,Rahul,Mobile,20000,2,40000
109,Priya,Tablet,15000,2,30000
110,Anil,Laptop,55000,1,55000


Saved records: 7


In [0]:
# Calculate total revenue by product

from pyspark.sql.functions import sum as spark_sum, count, col

product_sales = sales_df.groupBy("product").agg(
    spark_sum("total_amount").alias("total_revenue"),
    count("order_id").alias("total_orders")
)

display(
    product_sales.orderBy(
        col("total_revenue").desc()
    )
)


product,total_revenue,total_orders
Laptop,220000,3
Mobile,80000,2
Tablet,45000,2


In [0]:
# Run overall sales metrics

from pyspark.sql.functions import sum as spark_sum, avg, count

sales_df.select(
    count("order_id").alias("total_orders"),
    spark_sum("total_amount").alias("total_revenue"),
    avg("total_amount").alias("average_order_value")
).show()


+------------+-------------+-------------------+
|total_orders|total_revenue|average_order_value|
+------------+-------------+-------------------+
|           7|       345000|  49285.71428571428|
+------------+-------------+-------------------+

